# Model Benchmark: Neural Networks vs Sklearn Baselines + XGBoost

## What this notebook does
- Uses **consistent preprocessing** for all models: median imputation + StandardScaler, 70/15/15 split
- Runs an **expanded hyperparameter grid** for all three NNs
- Compares against Yefeng's sklearn baselines (LogReg, Decision Tree, Random Forest) + **XGBoost**

## Key fixes vs prior NN notebooks
| Issue | Before | Now |
|---|---|---|
| Missing value handling | `dropna` (lost 6 rows) | Median imputation (303 samples) |
| Feature scaling | **Not applied to NNs** | `StandardScaler` fit on train set |
| Loss function | `NLLLoss + log(softmax)` | `CrossEntropyLoss` (raw logits) |
| Architecture | No BatchNorm | BatchNorm after each hidden layer |
| Training | Fixed 150 epochs | Early stopping (patience=20, max 300) |
| Grid size | 36 combos | 54 combos per model (+ 162 for DualHead) |

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from itertools import product
import matplotlib.pyplot as plt
import warnings, json
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Imports OK')

## 1. Data Loading & Preprocessing

In [ ]:
import ssl, urllib.request, io
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

url = 'https://raw.githubusercontent.com/dataprofessor/data/master/heart-disease-cleveland.csv'
try:
    df = pd.read_csv(url)
except Exception:
    ctx = ssl._create_unverified_context()
    with urllib.request.urlopen(url, context=ctx) as r:
        df = pd.read_csv(io.BytesIO(r.read()))

df.columns = df.columns.str.strip()
df = df.replace('?', np.nan)
df['diagnosis'] = df['diagnosis'].astype(float)
feature_cols = [c for c in df.columns if c != 'diagnosis']

# Median imputation — keeps all 303 rows
imputer = SimpleImputer(strategy='median')
X_all = imputer.fit_transform(df[feature_cols].astype(float))
y_all = df['diagnosis'].values.astype(int)

print(f'Total samples after imputation: {len(X_all)}')
print(f'Class distribution: {dict(zip(*np.unique(y_all, return_counts=True)))}')

# 70 / 15 / 15 split
df2 = pd.DataFrame(X_all, columns=feature_cols)
df2['diagnosis'] = y_all
df2 = df2.sample(frac=1, random_state=SEED).reset_index(drop=True)

n = len(df2)
train_end = int(0.70 * n); val_end = int(0.85 * n)
train_df = df2.iloc[:train_end]
val_df   = df2.iloc[train_end:val_end]
test_df  = df2.iloc[val_end:]
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

X_tr_raw = train_df[feature_cols].values
X_va_raw = val_df[feature_cols].values
X_te_raw = test_df[feature_cols].values

scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr_raw)
X_va = scaler.transform(X_va_raw)
X_te = scaler.transform(X_te_raw)

y_tr_bin = (train_df['diagnosis'].values > 0).astype(int)
y_va_bin = (val_df['diagnosis'].values   > 0).astype(int)
y_te_bin = (test_df['diagnosis'].values  > 0).astype(int)

y_tr_mc = train_df['diagnosis'].values.astype(int)
y_va_mc = val_df['diagnosis'].values.astype(int)
y_te_mc = test_df['diagnosis'].values.astype(int)

## 2. Sklearn Baselines + XGBoost

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score
from xgboost import XGBClassifier

def bin_metrics(y_true, y_pred):
    return accuracy_score(y_true, y_pred), recall_score(y_true, y_pred, pos_label=1)

def mc_metrics(y_true, y_pred):
    return accuracy_score(y_true, y_pred), recall_score(y_true, y_pred, average='macro', zero_division=0)

# --- Binary ---
lr_m = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=SEED)
lr_m.fit(X_tr, y_tr_bin)
lr_acc, lr_rec = bin_metrics(y_te_bin, lr_m.predict(X_te))

dt_m = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=SEED)
dt_m.fit(X_tr, y_tr_bin)
dt_acc, dt_rec = bin_metrics(y_te_bin, dt_m.predict(X_te))

rf_m = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=SEED)
rf_m.fit(X_tr, y_tr_bin)
rf_acc, rf_rec = bin_metrics(y_te_bin, rf_m.predict(X_te))

xgb_b = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                       subsample=0.8, colsample_bytree=0.8,
                       eval_metric='logloss', random_state=SEED, verbosity=0)
xgb_b.fit(X_tr, y_tr_bin, eval_set=[(X_va, y_va_bin)], verbose=False)
xgb_b_acc, xgb_b_rec = bin_metrics(y_te_bin, xgb_b.predict(X_te))

# --- Multi-class ---
rf_mc = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=SEED)
rf_mc.fit(X_tr, y_tr_mc)
rf_mc_acc, rf_mc_rec = mc_metrics(y_te_mc, rf_mc.predict(X_te))

xgb_mc = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                        subsample=0.8, colsample_bytree=0.8,
                        objective='multi:softprob', num_class=5,
                        eval_metric='mlogloss', random_state=SEED, verbosity=0)
xgb_mc.fit(X_tr, y_tr_mc, eval_set=[(X_va, y_va_mc)], verbose=False)
xgb_mc_acc, xgb_mc_rec = mc_metrics(y_te_mc, xgb_mc.predict(X_te))

print(f'Binary:      LogReg acc={lr_acc:.3f} rec={lr_rec:.3f}  |  RF acc={rf_acc:.3f} rec={rf_rec:.3f}  |  XGB acc={xgb_b_acc:.3f} rec={xgb_b_rec:.3f}')
print(f'Multi-class: RF acc={rf_mc_acc:.3f} macro_rec={rf_mc_rec:.3f}  |  XGB acc={xgb_mc_acc:.3f} macro_rec={xgb_mc_rec:.3f}')

## 3. Neural Networks — Improved Architecture + Grid Search

Grid: `lr` ∈ {1e-2, 1e-3, 1e-4}, `hidden` ∈ {32, 64, 128}, `dropout` ∈ {0.0, 0.2, 0.3}, `weight_decay` ∈ {0.0, 1e-4}  
→ **54 combinations** per model; **162 combinations** for DualHeadNet (+ loss_weight ∈ {0.3, 0.5, 0.7})

In [ ]:
from sklearn.metrics import recall_score

Xtr_f = torch.tensor(X_tr, dtype=torch.float32)
Xva_f = torch.tensor(X_va, dtype=torch.float32)
Xte_f = torch.tensor(X_te, dtype=torch.float32)
ytr_bin_t = torch.tensor(y_tr_bin, dtype=torch.long)
yva_bin_t = torch.tensor(y_va_bin, dtype=torch.long)
yte_bin_t = torch.tensor(y_te_bin, dtype=torch.long)
ytr_mc_t  = torch.tensor(y_tr_mc,  dtype=torch.long)
yva_mc_t  = torch.tensor(y_va_mc,  dtype=torch.long)
yte_mc_t  = torch.tensor(y_te_mc,  dtype=torch.long)

D = X_tr.shape[1]
CE = nn.CrossEntropyLoss()
PATIENCE = 20
MAX_EP   = 300

class Net(nn.Module):
    def __init__(self, in_d, h, drop, out_d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(h, h//2), nn.BatchNorm1d(h//2), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(h//2, out_d),
        )
    def forward(self, x): return self.net(x)

class DualNet(nn.Module):
    def __init__(self, in_d, h, drop):
        super().__init__()
        self.bb = nn.Sequential(
            nn.Linear(in_d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(drop),
            nn.Linear(h, h//2), nn.BatchNorm1d(h//2), nn.ReLU(), nn.Dropout(drop),
        )
        self.bh = nn.Linear(h//2, 2)
        self.mh = nn.Linear(h//2, 5)
    def forward(self, x):
        f = self.bb(x); return self.bh(f), self.mh(f)

def val_acc(model, X, y):
    return (model(X).argmax(1) == y).float().mean().item()

def run_single(model, Xtr, ytr, Xva, yva, lr, wd):
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True)
    best, wait, best_state = 0.0, 0, None
    for _ in range(MAX_EP):
        model.train()
        for xb, yb in loader:
            opt.zero_grad(); CE(model(xb), yb).backward(); opt.step()
        model.eval()
        with torch.no_grad(): v = val_acc(model, Xva, yva)
        if v >= best:
            best=v; wait=0
            best_state={k: vv.clone() for k, vv in model.state_dict().items()}
        else:
            wait+=1
            if wait>=PATIENCE: break
    model.load_state_dict(best_state)
    return best

def run_dual(model, Xtr, ytr_b, ytr_m, Xva, yva_b, yva_m, lr, wd, lw):
    opt = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    loader = DataLoader(TensorDataset(Xtr, ytr_b, ytr_m), batch_size=32, shuffle=True)
    best_avg, wait, best_state, best_accs = 0.0, 0, None, (0,0)
    for _ in range(MAX_EP):
        model.train()
        for xb, yb, ym in loader:
            opt.zero_grad()
            ob, om = model(xb)
            (lw*CE(ob,yb)+(1-lw)*CE(om,ym)).backward(); opt.step()
        model.eval()
        with torch.no_grad():
            ob, om = model(Xva)
            ab=(ob.argmax(1)==yva_b).float().mean().item()
            am=(om.argmax(1)==yva_m).float().mean().item()
        avg=0.5*(ab+am)
        if avg>=best_avg:
            best_avg=avg; wait=0
            best_state={k:v.clone() for k,v in model.state_dict().items()}
            best_accs=(ab,am)
        else:
            wait+=1
            if wait>=PATIENCE: break
    model.load_state_dict(best_state)
    return best_accs

def nn_bin_metrics(model, Xte, yte_np):
    model.eval()
    with torch.no_grad(): preds = model(Xte).argmax(1).numpy()
    return accuracy_score(yte_np, preds), recall_score(yte_np, preds, pos_label=1)

def nn_mc_metrics(model, Xte, yte_np):
    model.eval()
    with torch.no_grad(): preds = model(Xte).argmax(1).numpy()
    return accuracy_score(yte_np, preds), recall_score(yte_np, preds, average='macro', zero_division=0)

lrs  = [1e-2, 1e-3, 1e-4]
hids = [32, 64, 128]
dos  = [0.0, 0.2, 0.3]
wds  = [0.0, 1e-4]
combos = list(product(lrs, hids, dos, wds))
print(f'Grid: {len(combos)} combos per single-head model')

In [ ]:
print(f'[BinaryNet] searching {len(combos)} configs...')
bin_res = []
for i, (lr, h, do, wd) in enumerate(combos):
    torch.manual_seed(SEED)
    m = Net(D, h, do, 2)
    va = run_single(m, Xtr_f, ytr_bin_t, Xva_f, yva_bin_t, lr, wd)
    te_acc, te_rec = nn_bin_metrics(m, Xte_f, y_te_bin)
    bin_res.append(dict(lr=lr, h=h, do=do, wd=wd, val=va, acc=te_acc, rec=te_rec, model=m))
    if (i+1) % 18 == 0: print(f'  {i+1}/{len(combos)} done')

bin_res.sort(key=lambda x: -x['val'])
best_b = bin_res[0]
nn_bin_acc, nn_bin_rec = best_b['acc'], best_b['rec']
print(f"\nBest: lr={best_b['lr']}, hidden={best_b['h']}, dropout={best_b['do']}, wd={best_b['wd']}")
print(f"Val={best_b['val']:.4f}  Test acc={nn_bin_acc:.4f}  Recall(1)={nn_bin_rec:.4f}")

In [ ]:
print(f'[MultiClassNet] searching {len(combos)} configs...')
mc_res = []
for i, (lr, h, do, wd) in enumerate(combos):
    torch.manual_seed(SEED)
    m = Net(D, h, do, 5)
    va = run_single(m, Xtr_f, ytr_mc_t, Xva_f, yva_mc_t, lr, wd)
    te_acc, te_rec = nn_mc_metrics(m, Xte_f, y_te_mc)
    mc_res.append(dict(lr=lr, h=h, do=do, wd=wd, val=va, acc=te_acc, rec=te_rec, model=m))
    if (i+1) % 18 == 0: print(f'  {i+1}/{len(combos)} done')

mc_res.sort(key=lambda x: -x['val'])
best_mc = mc_res[0]
nn_mc_acc, nn_mc_rec = best_mc['acc'], best_mc['rec']
print(f"\nBest: lr={best_mc['lr']}, hidden={best_mc['h']}, dropout={best_mc['do']}, wd={best_mc['wd']}")
print(f"Val={best_mc['val']:.4f}  Test acc={nn_mc_acc:.4f}  Macro recall={nn_mc_rec:.4f}")

In [ ]:
dual_combos = list(product(lrs, hids, dos, wds, [0.3, 0.5, 0.7]))
print(f'[DualHeadNet] searching {len(dual_combos)} configs...')
dual_res = []
for i, (lr, h, do, wd, lw) in enumerate(dual_combos):
    torch.manual_seed(SEED)
    m = DualNet(D, h, do)
    ab, am = run_dual(m, Xtr_f, ytr_bin_t, ytr_mc_t,
                      Xva_f, yva_bin_t, yva_mc_t, lr, wd, lw)
    m.eval()
    with torch.no_grad():
        ob, om = m(Xte_f)
        pb = ob.argmax(1).numpy(); pm = om.argmax(1).numpy()
    b_acc = accuracy_score(y_te_bin, pb); b_rec = recall_score(y_te_bin, pb, pos_label=1)
    m_acc = accuracy_score(y_te_mc,  pm); m_rec = recall_score(y_te_mc,  pm, average='macro', zero_division=0)
    dual_res.append(dict(lr=lr, h=h, do=do, wd=wd, lw=lw,
                         val_b=ab, val_m=am,
                         b_acc=b_acc, b_rec=b_rec,
                         m_acc=m_acc, m_rec=m_rec, model=m))
    if (i+1) % 54 == 0: print(f'  {i+1}/{len(dual_combos)} done')

dual_res.sort(key=lambda x: -(x['val_b']+x['val_m']))
best_d = dual_res[0]
nn_dual_bin_acc, nn_dual_bin_rec = best_d['b_acc'], best_d['b_rec']
nn_dual_mc_acc,  nn_dual_mc_rec  = best_d['m_acc'], best_d['m_rec']
print(f"\nBest: lr={best_d['lr']}, hidden={best_d['h']}, dropout={best_d['do']}, wd={best_d['wd']}, lw={best_d['lw']}")
print(f"Val  bin={best_d['val_b']:.4f}  mc={best_d['val_m']:.4f}")
print(f"Test binary:     acc={nn_dual_bin_acc:.4f}  recall(1)={nn_dual_bin_rec:.4f}")
print(f"Test multi-cls:  acc={nn_dual_mc_acc:.4f}  macro_recall={nn_dual_mc_rec:.4f}")

## 4. Benchmark Results

In [ ]:
bin_rows = [
    ('Logistic Regression',       'sklearn',  'Yefeng', lr_acc,           lr_rec),
    ('Decision Tree',             'sklearn',  'Yefeng', dt_acc,           dt_rec),
    ('Random Forest (balanced)',  'sklearn',  'Yefeng', rf_acc,           rf_rec),
    ('XGBoost',                   'boosting', 'New',    xgb_b_acc,        xgb_b_rec),
    ('BinaryNet NN',              'NN',       'Jacob',  nn_bin_acc,       nn_bin_rec),
    ('DualHeadNet – binary head', 'NN',       'Jacob',  nn_dual_bin_acc,  nn_dual_bin_rec),
]
bin_df = (pd.DataFrame(bin_rows, columns=['Model','Type','Author','Accuracy','Recall(1)'])
            .sort_values('Accuracy', ascending=False).reset_index(drop=True))
print('BINARY CLASSIFICATION (test set)')
print(bin_df[['Model','Author','Accuracy','Recall(1)']].to_string(index=False))

print()

mc_rows = [
    ('Random Forest (multi-class)',    'sklearn',  'Yefeng', rf_mc_acc,      rf_mc_rec),
    ('XGBoost (multi-class)',          'boosting', 'New',    xgb_mc_acc,     xgb_mc_rec),
    ('MultiClassNet NN',               'NN',       'Jacob',  nn_mc_acc,      nn_mc_rec),
    ('DualHeadNet – multi-class head', 'NN',       'Jacob',  nn_dual_mc_acc, nn_dual_mc_rec),
]
mc_df = (pd.DataFrame(mc_rows, columns=['Model','Type','Author','Accuracy','MacroRecall'])
           .sort_values('Accuracy', ascending=False).reset_index(drop=True))
print('MULTI-CLASS 0-4 (test set, macro recall)')
print(mc_df[['Model','Author','Accuracy','MacroRecall']].to_string(index=False))

In [ ]:
colour_map = {'sklearn': '#4C72B0', 'boosting': '#DD8452', 'NN': '#55A868'}

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Model Benchmark — Cleveland Heart Disease Dataset', fontsize=13, fontweight='bold')

def grouped_bar(ax, df, metric_col, title, xlim, xlabel):
    df_s = df.sort_values(metric_col)
    colours = [colour_map[t] for t in df_s['Type']]
    bars = ax.barh(df_s['Model'], df_s[metric_col], color=colours, edgecolor='white')
    ax.set_xlim(*xlim)
    ax.set_xlabel(xlabel)
    ax.set_title(title)
    for bar, val in zip(bars, df_s[metric_col]):
        ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                f'{val:.3f}', va='center', fontsize=8.5)

grouped_bar(axes[0,0], bin_df, 'Accuracy',  'Binary — Accuracy',        (0.55, 1.0),  'Test Accuracy')
grouped_bar(axes[0,1], bin_df, 'Recall(1)', 'Binary — Recall (class 1)', (0.45, 1.05), 'Recall (disease=1)')
grouped_bar(axes[1,0], mc_df,  'Accuracy',  'Multi-class — Accuracy',    (0.30, 0.75), 'Test Accuracy')
grouped_bar(axes[1,1], mc_df,  'MacroRecall','Multi-class — Macro Recall',(0.10, 0.55), 'Macro Recall')

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=colour_map['sklearn'],   label='sklearn (Yefeng)'),
                   Patch(facecolor=colour_map['boosting'],  label='XGBoost (new)'),
                   Patch(facecolor=colour_map['NN'],        label='Neural Network (Jacob)')]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.01))

plt.tight_layout()
plt.savefig('benchmark_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to benchmark_chart.png')

## 5. Discussion

### Binary classification
| Model | Accuracy | Recall (disease=1) |
|---|---|---|
| Logistic Regression | 0.848 | **0.857** |
| DualHeadNet – binary head | 0.826 | 0.810 |
| Random Forest (balanced) | 0.826 | 0.762 |
| XGBoost | 0.804 | 0.762 |
| BinaryNet NN | 0.761 | 0.714 |
| Decision Tree | 0.761 | 0.667 |

- **Logistic Regression is the best binary model** on both accuracy and recall — a well-regularised linear model generalises better than the others at n=303.
- **DualHeadNet has the best recall among the NNs** (0.810), edging out Random Forest, showing the shared backbone helps the binary head.
- Recall (class 1 = disease present) matters more than accuracy in clinical settings — false negatives are more costly than false positives.

### Multi-class (severity 0–4)
| Model | Accuracy | Macro Recall |
|---|---|---|
| MultiClassNet NN | **0.565** | **0.329** |
| Random Forest | 0.565 | 0.272 |
| XGBoost | 0.522 | 0.281 |
| DualHeadNet – multi-class head | 0.522 | 0.243 |

- **MultiClassNet has the best macro recall (0.329)**, meaning it covers minority severity classes better than Random Forest despite tying on accuracy.
- All macro recall scores are low because the dataset is heavily imbalanced (160 class-0 vs 13 class-4). Weighted loss or SMOTE would likely help.

### Note on variance
The test set is only 46 samples (1 sample ≈ 2.2%). Differences within ~0.04 are within noise — cross-validation would give more reliable estimates.

### Optimal NN hyperparameters
| Model | lr | hidden | dropout | wd | extra |
|---|---|---|---|---|---|
| BinaryNet | 0.001 | 128 | 0.3 | 0.0 | — |
| MultiClassNet | 0.001 | 64 | 0.0 | 1e-4 | — |
| DualHeadNet | 0.01 | 64 | 0.2 | 0.0 | loss_weight=0.7 |